# Part 1 · Bag of Words

> Kaggle *Bag of Words Meets Bags of Popcorn* 教程 Part 1 的 Python 3 复现。

**要解决的问题**：机器学习模型只接受数值向量，而影评是变长的字符串。
Bag of Words 给出最朴素的一种转换方式——**统计每个词出现了几次**。

```
"This movie is great, really great!"
            ↓  清洗 + 去停用词
["movie", "great", "really", "great"]
            ↓  按 5,000 词的词表计数
[0, 0, ..., 1(movie), ..., 2(great), ..., 1(really), ..., 0]
```

这条 5,000 维向量的每一维都对应一个**具体可读的词**，绝大多数是 0。
它完全丢掉了词序（"good not bad" 和 "bad not good" 向量相同），也不知道
`awful` 和 `terrible` 是近义词——这两个缺陷正是 Part 2 要解决的。


## Setup

纯 scikit-learn，CPU 即可。开 GPU 对本 Notebook **没有**加速效果。


In [ ]:
import csv
import re
from html import unescape
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

HTML_TAG = re.compile(r"<[^>]+>")
NON_LETTER = re.compile(r"[^a-zA-Z]")
SENTENCE_END = re.compile(r"(?<=[.!?])\s+")
STOP_WORDS = frozenset(ENGLISH_STOP_WORDS)
RANDOM_STATE = 42


def find_data_dir() -> Path:
    for candidate in [Path("/kaggle/input/word2vec-nlp-tutorial"), Path("data"), Path("../data")]:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("请用 Add Input 挂载 word2vec-nlp-tutorial 竞赛数据")


def read_tsv(data_dir: Path, stem: str) -> pd.DataFrame:
    # quoting=QUOTE_NONE：影评正文含大量引号，按默认规则解析会把字段粘连。
    for suffix in (".tsv", ".tsv.zip"):
        path = data_dir / f"{stem}{suffix}"
        if path.exists():
            return pd.read_csv(path, header=0, delimiter="\t", quoting=csv.QUOTE_NONE)
    raise FileNotFoundError(f"{data_dir} 下找不到 {stem}")


def review_to_words(raw_review, remove_stopwords: bool = True) -> list[str]:
    text = NON_LETTER.sub(" ", HTML_TAG.sub(" ", unescape(str(raw_review))))
    words = text.lower().split()
    if remove_stopwords:
        return [w for w in words if w not in STOP_WORDS]
    return words


def clean_review(raw_review, remove_stopwords: bool = True) -> str:
    return " ".join(review_to_words(raw_review, remove_stopwords))


DATA_DIR = find_data_dir()
print("数据目录:", DATA_DIR)


## 1 · 读取数据

Kaggle 把竞赛文件挂载为 `.tsv.zip`，`pandas` 依据后缀自动解压，不用手动 unzip。


In [ ]:
train = read_tsv(DATA_DIR, "labeledTrainData")
test = read_tsv(DATA_DIR, "testData")

print(f"训练集 {train.shape}，测试集 {test.shape}")
print(train["sentiment"].value_counts().sort_index().to_dict())
train.head(3)


In [ ]:
# 数据契约检查：先确认列名和取值符合预期，再往下走。
assert {"id", "sentiment", "review"} <= set(train.columns)
assert train["review"].notna().all()
assert set(train["sentiment"].unique()) == {0, 1}
print("输入校验通过")


## 2 · 清洗文本

影评是从网页抓来的，含 `<br />` 标签和 HTML 实体。清洗四步：反转义 →
去标签 → 只保留字母 → 小写化并去停用词。

> 原教程用 `BeautifulSoup` + `nltk.corpus.stopwords`。这里换成标准库正则和
> sklearn 内置停用词表，效果等价，且不需要 `nltk.download()`——Kaggle
> Notebook 默认断网，下载会直接失败。


In [ ]:
raw = train.loc[0, "review"]
print("清洗前:", raw[:280], "\n")
print("清洗后:", clean_review(raw)[:280])


In [ ]:
clean_train = train["review"].map(clean_review)
clean_test = test["review"].map(clean_review)
print(f"平均长度：{train['review'].str.split().str.len().mean():.0f} 词 "
      f"→ {clean_train.str.split().str.len().mean():.0f} 词")


## 3 · 先划分，再建词表

顺序很关键：**必须先切出验证集，再只用训练分片 `fit` 词表**。
如果先在全量数据上建词表，验证集的词频分布就泄漏进了特征空间，
指标会虚高。`stratify` 保证两边正负比例一致。


In [ ]:
from sklearn.model_selection import train_test_split

train_text, valid_text, train_y, valid_y = train_test_split(
    clean_train, train["sentiment"],
    test_size=0.2, random_state=RANDOM_STATE, stratify=train["sentiment"],
)
print(f"训练 {len(train_text):,} / 验证 {len(valid_text):,}")


## 4 · 构造 Bag of Words 特征

`CountVectorizer` 取词频最高的 5,000 个词建表，输出**稀疏矩阵**。

> 原教程调用了 `.toarray()`。25,000 × 5,000 的 float64 稠密矩阵约 1 GB，
> 而实际非零元素只占 ~1.4%。保留 `scipy.sparse` 格式，随机森林可以直接吃。


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=5000)
train_x = vectorizer.fit_transform(train_text)
valid_x = vectorizer.transform(valid_text)

density = train_x.nnz / (train_x.shape[0] * train_x.shape[1])
print(f"特征矩阵 {train_x.shape}，非零元素占比 {density:.2%}")
print(f"稠密化需要 {train_x.shape[0] * train_x.shape[1] * 8 / 1e9:.2f} GB，"
      f"稀疏存储只需 {train_x.data.nbytes / 1e6:.1f} MB")


In [ ]:
vocab = vectorizer.get_feature_names_out()
counts = np.asarray(train_x.sum(axis=0)).ravel()
top = pd.DataFrame({"word": vocab, "count": counts}).nlargest(15, "count")
print("词表片段:", vocab[:12].tolist())
top.reset_index(drop=True)


## 5 · 训练随机森林


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)
model.fit(train_x, train_y)
print("训练完成")


## 6 · 在留出集上评估

只看 accuracy 是不够的。加上 ROC-AUC（模型给正样本更高分的能力）和
混淆矩阵（错在哪一边），才能判断模型是真的学到了东西还是在猜。


In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)

predictions = model.predict(valid_x)
probabilities = model.predict_proba(valid_x)[:, 1]

print(f"Accuracy: {accuracy_score(valid_y, predictions):.4f}")
print(f"ROC-AUC : {roc_auc_score(valid_y, probabilities):.4f}\n")
print(classification_report(valid_y, predictions, digits=4,
                            target_names=["negative", "positive"]))


In [ ]:
import matplotlib.pyplot as plt

matrix = confusion_matrix(valid_y, predictions)
fig, ax = plt.subplots(figsize=(4.4, 3.8))
ax.imshow(matrix, cmap="Blues")
for (i, j), value in np.ndenumerate(matrix):
    ax.text(j, i, f"{value:,}", ha="center", va="center",
            color="white" if value > matrix.max() / 2 else "black")
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=["negative", "positive"], yticklabels=["negative", "positive"],
       xlabel="预测", ylabel="真实", title="Confusion Matrix · BoW + RF")
plt.tight_layout()
plt.show()


## 7 · 哪些词最有区分力

随机森林的 `feature_importances_` 可以直接映射回具体的词——这是 BoW
相对稠密 Embedding 的一个真实优势：**特征可解释**。


In [ ]:
importance = (pd.DataFrame({"word": vocab, "importance": model.feature_importances_})
              .nlargest(20, "importance").sort_values("importance"))

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.barh(importance["word"], importance["importance"], color="#4c72b0")
ax.set(xlabel="feature importance", title="最具区分力的 20 个词")
plt.tight_layout()
plt.show()


## 8 · 全量重训并生成提交文件

验证集的作用是选方法。方法定了之后，用**全部** 25,000 条标注数据重训，
让模型见到尽可能多的样本，再预测测试集。


In [ ]:
final_vectorizer = CountVectorizer(max_features=5000)
final_train_x = final_vectorizer.fit_transform(clean_train)
final_test_x = final_vectorizer.transform(clean_test)

final_model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)
final_model.fit(final_train_x, train["sentiment"])

submission = pd.DataFrame({"id": test["id"], "sentiment": final_model.predict(final_test_x)})
output = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")
submission.to_csv(output, index=False, quoting=csv.QUOTE_NONE)
print(f"已写出 {output}（{len(submission):,} 行）")
submission.head()


## 小结

| | |
|---|---|
| 向量维度 | 5,000（= 词表大小） |
| 每一维的含义 | 某个具体词的出现次数 |
| 稀疏度 | ~98.6% 是 0 |
| 是否懂近义词 | 不懂，`awful` 与 `terrible` 完全正交 |
| 是否懂词序 | 不懂 |

**下一步（Part 2）**：训练 Word2Vec，把每个词压成 300 维稠密向量，
让近义词在空间里真正靠在一起。
